# Paper-ball YOLO11n training (Roboflow "paper" dataset, cleaned)

Trains a small YOLO11n paper-ball detector on a Colab GPU and exports it as ONNX for `camera_distance.py`.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU`.
2. Have `dataset_fixed.zip` ready (made by `fix_dataset.py` in `Training/paperball_roboflow/`). The data comes from Roboflow Universe under **CC BY 4.0**: credit the author if you share the model or the data.

Then run the cells from top to bottom. Training takes roughly 10-25 minutes on a T4.

In [ ]:
!nvidia-smi -L || echo "NO GPU: use Runtime > Change runtime type > T4 GPU, then run again"
!pip -q install ultralytics onnx onnxslim

## 1. Upload the dataset zip
Choose `dataset_fixed.zip` (about 80 MB, 273 training and 30 validation pictures). To use Google Drive instead, see the commented lines.

In [ ]:
import os, zipfile, pathlib
from google.colab import files

# Option B (Drive): put dataset_fixed.zip in My Drive, then uncomment:
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/dataset_fixed.zip /content/

if not os.path.exists('/content/dataset_fixed.zip'):
    uploaded = files.upload()          # pick dataset_fixed.zip in the dialog

root = pathlib.Path('/content/dataset_fixed')
zipfile.ZipFile('/content/dataset_fixed.zip').extractall(root)

# Write a data.yaml with absolute paths so Ultralytics finds the pictures
import yaml
cfg = {'path': str(root), 'train': 'train/images', 'val': 'valid/images', 'names': {0: 'paper'}}
(root / 'data.yaml').write_text(yaml.safe_dump(cfg))
for split in ('train', 'valid'):
    print(split, len(list((root / split / 'images').glob('*'))), 'pictures')

## 2. Train
Change `EPOCHS` if you like (more epochs = slower, usually a bit better until it stops improving).

In [ ]:
from ultralytics import YOLO

EPOCHS = 80
model = YOLO('yolo11n.pt')             # downloads the small pretrained model
results = model.train(
    data=str(root / 'data.yaml'), epochs=EPOCHS, imgsz=640, batch=16, device=0,
    project='/content/runs', name='roboflow-paperball', exist_ok=True,
    flipud=0.0, plots=False, patience=30,     # stops early if it has not improved for 30 epochs
)
best = '/content/runs/roboflow-paperball/weights/best.pt'
print('best weights:', best)

## 3. Check the result
A high `mAP50` on the validation pictures is expected, because they come from the same photo sessions as the training pictures. The real test is your own camera.

In [ ]:
m = YOLO(best)
metrics = m.val(data=str(root / 'data.yaml'), imgsz=640)
print(f'mAP50 {metrics.box.map50:.3f}   mAP50-95 {metrics.box.map:.3f}')

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
paths = sorted(glob.glob(str(root / 'valid/images/*')))[:8]
preds = m.predict(paths, conf=0.5, imgsz=640, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.ravel(), preds):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Export and download
This gives `roboflow-paperball.pt` and `roboflow-paperball.onnx`. Put both into `Training/paperball_roboflow/` on your PC.

In [ ]:
import shutil
shutil.copy2(best, '/content/roboflow-paperball.pt')
onnx_path = YOLO('/content/roboflow-paperball.pt').export(format='onnx', imgsz=640, simplify=True, opset=12)
shutil.copy2(onnx_path, '/content/roboflow-paperball.onnx')
files.download('/content/roboflow-paperball.pt')
files.download('/content/roboflow-paperball.onnx')

## 5. Try it on your camera (on your PC)

```powershell
cd C:\Users\HP\AndroidStudioProjects\LitterVision\Embedded\esp32-cam\scripts
py camera_distance.py --model ..\..\..\Training\paperball_roboflow\roboflow-paperball.onnx --rotate ccw
```

Compare it with `yolo-temp` (the default model of that script) on the same scene.